In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
pd.option_context('display.max_rows', None)
pd.option_context('display.max_columns', None)

##### Load the datasets

In [2]:
filename = Path("Clariti Consumption/LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq")
df = pd.read_parquet(filename, engine='fastparquet')

# Remove NMIs 6102507141 and VAAA003225
print("Original number of NMIs: " + str(len(df.columns)-1))
df.drop(columns=["6102507141 consumption","VAAA003225 consumption"], inplace=True)
print("New number of NMIs: "  + str(len(df.columns)-1))

Original number of NMIs: 101
New number of NMIs: 99


##### 1. Understand the structure of the dataset

First, make sure you clearly understand what the data represents. At this stage, please confirm and document:

- what each row represents
- what each NMI column represents
- the time frequency of the data
- the total time span covered
- the number of NMI series available

Aim: to make sure we are all working with the same understanding of the dataset. At the NMI level, each NMI column should be treated as one separate half-hourly consumption time series.

##### 2. Check the time index and general data quality

Please verify that:

- the date column is correctly parsed as datetime
- the timestamps are sorted
- the data is recorded at regular 30-minute intervals
- there are no duplicate timestamps
- there are no missing timestamps across the overall dataset

Also check, for each NMI:

- missing values
- zero values
- negative values
- basic summary statistics such as mean, standard deviation, minimum, and maximum
- the date of the first non-zero reading and the date of the last non-zero reading for each NMI

Aim: to confirm that the basic structure of the data is sound and to identify obvious quality issues early.

In [3]:
df['date'] = pd.to_datetime(df['date'])
df.sort_values(by='date', ascending=True)
print(df[df['date'].duplicated()]['date'])

Series([], Name: date, dtype: datetime64[us])


No duplicate timestamp.

In [4]:
df['time_gap'] = df['date'].diff()
df['time_gap'].value_counts()

time_gap
0 days 00:30:00    231839
Name: count, dtype: int64

All intervals are 30 minutes. No missing timestamp.

In [5]:
del df['time_gap']

In [6]:
NMI_list = df.columns.tolist()[1:]
data = {}
for col in NMI_list:
    first_date = min(df[df[col]!=0]['date'])
    last_date = max(df[df[col]!=0]['date'])
    filtered_col = df[((df['date']>=first_date) & (df['date']<=last_date))][[col]]
    
    zeroConsumption = len(filtered_col[filtered_col[col]==0])
    percentZeroConsumption = round(zeroConsumption*100 / len(filtered_col),2)
    negativeConsumption = len(filtered_col[filtered_col[col]<0])

    summarystats = filtered_col.describe()
    
    data[col.split(' ')[0]] = {
        'missing values':df[col].isna().sum(),
        'first date': first_date,
        'last date': last_date,
        '# of zero': zeroConsumption,
        '% of zero': percentZeroConsumption,
        '# of negative': negativeConsumption,
        'min': summarystats.loc['min', col],
        'max': summarystats.loc['max', col],
        'mean': round(summarystats.loc['mean', col],3),
        'std dev': round(summarystats.loc['std', col],3)
    }

    try:
        df.loc[df['date'] < first_date, col] = np.nan
    except:
        continue

summary_df = pd.DataFrame.from_dict(data, orient='index')
summary_df

,missing values,first date,last date,# of zero,% of zero,# of negative,min,max,mean,std dev
6102000812,0,2013-01-01,2026-03-23 23:30:00,203,0.09,0,0.000,90.816,27.466,9.893
6102002302,0,2013-01-01,2026-03-23 23:30:00,3446,1.49,0,0.000,113.759,18.860,15.006
6102005454,0,2013-01-01,2026-03-23 23:30:00,145,0.06,0,0.000,145.700,66.030,18.446
6102005592,0,2013-01-01,2026-03-23 23:30:00,191,0.08,0,0.000,372.240,53.571,49.795
6102009742,0,2013-01-01,2026-03-23 23:30:00,196,0.08,0,0.000,16.672,6.675,1.617
...,...,...,...,...,...,...,...,...,...,...
VAAA004066,0,2025-01-01,2026-03-23 23:30:00,0,0.00,0,0.248,14.192,0.341,0.278
VCCCAE0035,0,2013-01-01,2026-03-23 23:30:00,353,0.15,0,0.000,322.208,122.177,32.398
VCCCBC0096,0,2013-01-01,2026-03-23 23:30:00,1666,0.72,0,0.000,89.205,27.536,9.774
VCCCSC0045,0,2013-01-01,2026-03-23 23:30:00,5731,2.47,0,0.000,118.336,26.705,16.944


##### 3. Classify NMI series by usability

Using the zero percentages and general summary statistics, classify the NMI series into groups such as:

- active
- mostly active
- intermittent
- mostly inactive
- dead

Aim: to distinguish between strong forecasting candidates and meters that are largely inactive or unsuitable for primary analysis.

Please interpret what these classes mean. For example, a mostly inactive or dead NMI may not be useful for forecasting, while active and mostly active NMIs should be prioritised.

In [7]:
conditions = [(summary_df['% of zero'] < 1), (summary_df['% of zero'] < 10), (summary_df['% of zero'] < 50), (summary_df['% of zero'] < 100) ]
status = ['Active', 'Mostly Active', 'Intermittent', 'Mostly inactive']
summary_df['Status'] = np.select(conditions, status, default='Dead')
summary_df.loc[summary_df['first date']==summary_df['last date'],'Status'] = 'Dead'
summary_df

,missing values,first date,last date,# of zero,% of zero,# of negative,min,max,mean,std dev,Status
6102000812,0,2013-01-01,2026-03-23 23:30:00,203,0.09,0,0.000,90.816,27.466,9.893,Active
6102002302,0,2013-01-01,2026-03-23 23:30:00,3446,1.49,0,0.000,113.759,18.860,15.006,Mostly Active
6102005454,0,2013-01-01,2026-03-23 23:30:00,145,0.06,0,0.000,145.700,66.030,18.446,Active
6102005592,0,2013-01-01,2026-03-23 23:30:00,191,0.08,0,0.000,372.240,53.571,49.795,Active
6102009742,0,2013-01-01,2026-03-23 23:30:00,196,0.08,0,0.000,16.672,6.675,1.617,Active
...,...,...,...,...,...,...,...,...,...,...,...
VAAA004066,0,2025-01-01,2026-03-23 23:30:00,0,0.00,0,0.248,14.192,0.341,0.278,Active
VCCCAE0035,0,2013-01-01,2026-03-23 23:30:00,353,0.15,0,0.000,322.208,122.177,32.398,Active
VCCCBC0096,0,2013-01-01,2026-03-23 23:30:00,1666,0.72,0,0.000,89.205,27.536,9.774,Active
VCCCSC0045,0,2013-01-01,2026-03-23 23:30:00,5731,2.47,0,0.000,118.336,26.705,16.944,Mostly Active


In [8]:
summary_df.to_csv('EDA_SummaryByNMI.csv')

In [9]:
summary_df['Status'].value_counts()

Status
Active             81
Mostly Active      14
Mostly inactive     2
Intermittent        1
Dead                1
Name: count, dtype: int64

In [10]:
NMI_shortlist = summary_df[((summary_df['Status']=='Active') | (summary_df['Status']=='Mostly Active'))].index
print(f'Number of shortlisted NMIs: {len(NMI_shortlist)}')
NMI_shortlist

Number of shortlisted NMIs: 95


Index(['6102000812', '6102002302', '6102005454', '6102005592', '6102009742',
       '6102009743', '6102009744', '6102023971', '6102038376', '6102046251',
       '6102047562', '6102079015', '6102126219', '6102136796', '6102159807',
       '6102241120', '6102253330', '6102284820', '6102329966', '6102332526',
       '6102479831', '6102548873', '6102573328', '6102798810', '6102823324',
       '6103002422', '6103004482', '6103005867', '6103005869', '6103009639',
       '6103010081', '6103010326', '6103011168', '6103015873', '6103022015',
       '6103022017', '6103022018', '6103029662', '6103029663', '6103031269',
       '6103031796', '6103054578', '6103054611', '6103055142', '6103055392',
       '6103055643', '6103056620', '6103056621', '6103056622', '6103056625',
       '6103063019', '6103063020', '6103065120', '6103065121', '6103065471',
       '6103066694', '6103067996', '6103068525', '6103077259', '6203397519',
       '6203397522', '6203848319', '6203949247', 'VAAA000057', 'VAAA000173',

##### 4. Merge the NMI series with the mapping file

For the active and mostly active NMIs, merge them with the mapping file and identify:

- the location or site name
- the location code
- whether the NMI maps cleanly to one location or appears across multiple records

Then create a simple interpretation grouping such as:

- clean single-location NMI
- shared or multi-location NMI
- duplicate mapping rows, needs review

Aim: to understand what each usable NMI actually represents in operational terms, and to distinguish clean modelling candidates from more ambiguous shared meters.

Please interpret whether the NMI looks like a simple stand-alone unit or something more complex such as a shared supply, billed total, or subsystem.

##### 5. Examine the main behavioural patterns of shortlisted clean NMIs

For a shortlist of clean single-location NMIs, plot and interpret:

- the full time series
- the average daily load shape
- weekday versus weekend load shape
- monthly average consumption

Aim: to understand the main operational behaviour of each series and identify whether there are clear daily, weekly, and seasonal patterns.

Please interpret whether the NMI appears to have a strong daily cycle, a strong weekday versus weekend difference, and clear monthly or seasonal variation.

##### 6. Structural break checks

For each shortlisted NMI, examine whether the behaviour of the series changes over time. In particular, please look for:

- sudden upward or downward shifts in the level of consumption
- noticeable changes in variability
- unusual periods where the pattern seems different from the rest of the history

A good way to do this is by plotting:

- the full half-hourly time series
- daily total consumption over time
- rolling mean and rolling standard deviation

Aim: to understand whether the series is stable across the full history, or whether there are major regime changes that may affect forecasting.

Please interpret the results. For example, if you find a large drop around a certain year, explain whether this may reflect a genuine operational change, a shutdown period, or a data issue.

##### 7. Zero-value pattern checks

In addition to the overall percentage of zeros, please check how zeros appear in the shortlisted series:

- are they isolated points?
- do they occur in long consecutive runs?
- do they occur around specific dates?

Please identify the longest zero runs and report their dates.

Aim: to distinguish between small random issues and more meaningful periods such as outages, closures, or possible recording problems.

Please interpret whether these zero periods look operationally meaningful or suspicious.

In [23]:
data = {}
for nmi in NMI_shortlist:
    data[nmi] = df[df[nmi + ' consumption']==0]['date'].reset_index(drop=True)
    
zero_df = pd.DataFrame.from_dict(data)

data = {}
run_count = 1

for nmi in NMI_shortlist:
    zero_df[nmi + ' gap'] = zero_df[nmi].diff()/ pd.Timedelta(minutes=30)
    data[nmi + ' range'] = []
    data[nmi + ' run count'] = []
    for index, row in zero_df.iterrows():
        if index == 0:
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
        elif row[nmi + ' gap'] == 1:
            last_date = zero_df.loc[index,nmi]
            run_count += 1
        else:
            data[nmi + ' range'].append(str(first_date) + ' to ' + str(last_date))
            data[nmi + ' run count'].append(run_count)
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
            run_count = 1
            if row[nmi + ' gap'] != row[nmi + ' gap']:
                break
        
zero_df = pd.DataFrame.from_dict(data,orient='index').transpose()

In [24]:
data = {}
for nmi in NMI_shortlist:
    range_count = zero_df[nmi + ' range'].count()
    max_run = zero_df[nmi + ' run count'].max()
    date_max_run = zero_df[zero_df[nmi + ' run count']==max_run][nmi + ' range'].tolist()
    date_max_run.append(None)

    mode_run = zero_df[nmi + ' run count'].mode()[0]
    date_mode_run = zero_df[zero_df[nmi + ' run count']==mode_run][nmi + ' range'].tolist()
    date_mode_run.append(None)
    
    if range_count < 100:
        if max_run <= 24:
            group = 1
        else:
            group = 2
    else:
        if max_run <= 24:
            group = 3
        else:
            group = 4

    data[nmi] = {
        'group': group,
        'count of zero ranges' : range_count,
        'max zero runs' : max_run,
        'date of max zero runs (1)' : date_max_run[0],
        'date of max zero runs (2)' : date_max_run[1],
        'mode zero runs' : mode_run,
        'date of mode zero runs (1)' : date_mode_run[0],
        'date of mode zero runs (2)' : date_mode_run[1]
    }

zero_df = pd.DataFrame.from_dict(data,orient='index')
zero_df.sort_values(by='group', ascending=True)
zero_df.to_csv('EDA_zero-value pattern checks.csv')
zero_df

##### 8. Daily totals and yearly comparisons

Please aggregate each shortlisted NMI to daily total consumption and analyse:

- how the daily totals move over time
- how the mean daily total changes by year
- whether some years look very different from others

Also compare the average daily load shape by year rather than only across the full sample.

Aim: to assess whether the load pattern is stable through time, and whether the same daily structure is preserved across years.

Please interpret whether the yearly patterns appear stable, gradually changing, or show a clear break.

##### 9. Lag structure and autocorrelation checks

For each shortlisted NMI, please examine how strongly current consumption is related to past consumption. Please calculate lag correlations for:

- lag 1 half-hour
- lag 2 half-hours
- lag 48 half-hours (same time previous day)
- lag 336 half-hours (same time previous week)

Present these as a summary table rather than scatter plots at this stage. A one-sentence interpretation per NMI is sufficient.

Aim: to identify whether the series has strong short-term, daily, and weekly dependence, which will later guide feature construction for forecasting models.

Please interpret which lag effects are strongest and what that suggests about the underlying consumption pattern.

##### 10. Distribution and outlier checks

For each shortlisted NMI, please inspect the distribution of consumption values using:

- histograms
- boxplots
- outlier counts based on the IQR rule

Please focus on the shape of the distribution and whether outliers are isolated spikes or cluster in particular periods. You do not need to repeat the basic summary statistics from Step 2.

Aim: to understand whether the series is relatively stable, highly skewed, or affected by unusual peaks that may need to be flagged later.

Please interpret whether the outliers seem likely to reflect real high-demand events or possible data problems.

##### 11. Weather and energy correlation

Using the weather dataset provided, please carry out the following:

- reshape the weather data from long format to wide format, with one column per weather variable
- resample from 15-minute to 30-minute intervals to match the energy data
- merge with the total campus consumption for the overlapping period (October 2025 to March 2026)
- compute correlations between each weather variable and total consumption
- plot a correlation heatmap
- plot the air temperature versus total consumption scatter, and the average consumption per temperature bin
- add Heating Degree Days (HDD) and Cooling Degree Days (CDD) using a base temperature of 18 degrees Celsius

Please note that the weather data only covers approximately five months of the thirteen-year energy dataset. This limits how much we can say about weather relationships across the full history. However, this analysis is still required by the project brief and gives us an important indication of how temperature and other variables drive demand.

Aim: to quantify the relationship between weather and energy consumption, and to identify which weather variables are most important for later forecasting models.

Please interpret which variables show the strongest relationship with consumption and whether the expected U-shaped temperature effect is visible.

##### 12. Cross-NMI correlation check

For the shortlisted active and mostly active NMIs, please compute a pairwise correlation matrix of their half-hourly consumption series and identify any pairs with a correlation above 0.95.

If any two NMIs are nearly perfectly correlated, this may indicate that they are measuring the same physical supply point, or that one is a sub-meter of the other. Flag these pairs for review before modelling.

Aim: to avoid double-counting energy in any site-level or campus-level aggregation, and to ensure that our shortlisted series are genuinely independent.

Please interpret whether any flagged pairs appear to be duplicates or sub-meters based on their mapping file entries.

In [38]:
correlation_matrix = df.drop(columns=['date']).corr()

data = {}
for i, nmi in enumerate(correlation_matrix.columns.tolist()):
    data[nmi] = correlation_matrix[((correlation_matrix[nmi]>=0.95)&(nmi!=correlation_matrix.index))].index

correlated_df = pd.DataFrame.from_dict(data,orient='index')
correlated_df.dropna(how='all', inplace=True)
correlated_df.to_csv('EDA_cross-nmi correlation check.csv')
correlated_df

,0
6102009743 consumption,6102009744 consumption
6102009744 consumption,6102009743 consumption


Only one flagged pair--they are located in the same building, readings are not duplicates